# Water Technician & Collection Purity Demo

This notebook is structured for a 15-minute VS Code + GitHub Copilot demo across the data science process: **define**, **clean**, **hypothesize**, **measure**, and **report**.

## 1) Define the business process

Business question: *Which sites and technicians are associated with lower water purity, and where should operational mitigation be prioritized?*

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('../data/water_technicians_collections.csv', parse_dates=['collection_date'])
df.head()

## 2) Get and clean data

Quick checks for missing values, typing, and outliers that may affect downstream analysis.

In [ ]:
df.info()

missing = df.isna().sum()
display(missing[missing > 0])

numeric_cols = ['flow_lpm', 'turbidity_ntu', 'ph', 'conductivity_us_cm', 'purity_score']
clean_df = df.copy()
clean_df[numeric_cols] = clean_df[numeric_cols].apply(pd.to_numeric, errors='coerce')
clean_df = clean_df.dropna(subset=numeric_cols)
clean_df.head()

## 3) Hypothesize

Hypothesis: *Higher turbidity and conductivity are associated with lower purity scores, especially at canal intake points.*

In [ ]:
clean_df[['turbidity_ntu', 'conductivity_us_cm', 'purity_score']].corr()

## 4) Measure

Aggregate by site and technician to identify where intervention has the highest expected impact.

In [ ]:
site_summary = (
    clean_df.groupby(['site_id', 'site_name', 'source_type'], as_index=False)
    .agg(avg_purity=('purity_score', 'mean'), alerts=('contamination_flag', 'sum'), samples=('collection_id', 'count'))
    .sort_values(['avg_purity', 'alerts'])
)

tech_summary = (
    clean_df.groupby(['technician_id', 'technician_name'], as_index=False)
    .agg(avg_purity=('purity_score', 'mean'), alerts=('contamination_flag', 'sum'), samples=('collection_id', 'count'))
    .sort_values(['avg_purity', 'alerts'])
)

display(site_summary.head(10))
display(tech_summary)

In [ ]:
plt.figure(figsize=(8, 4))
sns.scatterplot(data=clean_df, x='turbidity_ntu', y='purity_score', hue='source_type')
plt.title('Purity vs Turbidity by Source Type')
plt.tight_layout()
plt.show()

## 5) Report

Use this short narrative in your stakeholder readout:

- Canal sites have the lowest average purity and the highest contamination alerts.
- Industrial Gate is the most persistent hotspot and should remain under active mitigation.
- Turbidity shows a clear inverse relationship with purity and is a practical leading indicator for field alerts.